In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/ieee-fraud-detection/sample_submission.csv
/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv
/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv


In [2]:
from my_preprocessing_classes import CategoricalEncoder, reduce_mem_usage, TimeFeaturesEncoder, GroupAggregationTransformer, DropMissingFeatures

/kaggle/input/competitions/ieee-fraud-detection/sample_submission.csv
/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv
/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv


In [3]:
!pip install dagshub
!pip install mlflow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.1/273.1 kB 7.5 MB/s eta 0:00:00ta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.3/207.3 kB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 8.6 MB/s eta 0:00:00
  Attempting uninstall: dacite
    Found existing installation: dacite 1.9.2
    Uninstalling dacite-1.9.2:
      Successfully uninstalled dacite-1.9.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ydata-profiling 4.18.1 requires dacite<2,>=1.9, but you have dacite 1.6.0 which is incompatible.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 1.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 3.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.0 MB/s eta 0:00:00
   ━━━━━━

In [4]:
import dagshub
import mlflow
dagshub.init(repo_owner='mkekn23', repo_name='IEEE-CIS-Fraud-Detection', mlflow=True)

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=e1ab8fc1-078a-4311-8711-3dbaeb7d61ad&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=0f50c66b3aa77651dd0019a727c90a6ad7625de62680bbdef3911b8429ca0577




Accessing as mkekn23

Initialized MLflow to track repo "mkekn23/IEEE-CIS-Fraud-Detection"

Repository mkekn23/IEEE-CIS-Fraud-Detection initialized!

In [5]:
df_identity= pd.read_csv("/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv")
df_transaction = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv')
df = pd.merge(df_transaction, df_identity, on='TransactionID', how='left')

In [6]:
del df_transaction, df_identity

In [7]:
df.shape

(590540, 434)

In [8]:
X = df.drop('isFraud', axis=1)
y = df['isFraud']

X = reduce_mem_usage(X)
y = y.astype('int8') 
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

Memory usage of dataframe is 1950.87 MB


/kaggle/usr/lib/notebooks/mariamkeknadze/my_preprocessing_classes/my_preprocessing_classes.py:41: RuntimeWarning: overflow encountered in cast
  if c_min > np.finfo(np.float16).min and c_max < np.finfo(np.float16).max:
/kaggle/usr/lib/notebooks/mariamkeknadze/my_preprocessing_classes/my_preprocessing_classes.py:41: RuntimeWarning: overflow encountered in cast
  if c_min > np.finfo(np.float16).min and c_max < np.finfo(np.float16).max:
/kaggle/usr/lib/notebooks/mariamkeknadze/my_preprocessing_classes/my_preprocessing_classes.py:41: RuntimeWarning: overflow encountered in cast
  if c_min > np.finfo(np.float16).min and c_max < np.finfo(np.float16).max:
/kaggle/usr/lib/notebooks/mariamkeknadze/my_preprocessing_classes/my_preprocessing_classes.py:41: RuntimeWarning: overflow encountered in cast
  if c_min > np.finfo(np.float16).min and c_max < np.finfo(np.float16).max:
/kaggle/usr/lib/notebooks/mariamkeknadze/my_preprocessing_classes/my_preprocessing_classes.py:41: RuntimeWarning: overflow e

Memory usage after optimization is: 524.99 MB
Decreased by 73.1%


# Training

In [9]:
from sklearn.metrics import roc_auc_score, recall_score, precision_score, f1_score, average_precision_score

def calculate_fraud_metrics(y_true, y_pred, y_probs):
    return {
        "auc_roc": roc_auc_score(y_true, y_probs),
        "pr_auc": average_precision_score(y_true, y_probs), # მნიშვნელოვანია დისბალანსისას
        "recall": recall_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred),
        "f1": f1_score(y_true, y_pred)
    }

import mlflow

def run_experiment(pipeline, run_name, params=None):
    with mlflow.start_run(run_name=run_name):
        if params:
            mlflow.log_params(params)
        
        pipeline.fit(X_train, y_train)
        
        y_pred = pipeline.predict(X_test)
        y_probs = pipeline.predict_proba(X_test)[:, 1]
        
        metrics = calculate_fraud_metrics(y_test, y_pred, y_probs)
        

        mlflow.log_metrics(metrics)
        
        mlflow.sklearn.log_model(pipeline, "model")
        
        print(f"Run '{run_name}' completed. AUC: {metrics['auc_roc']:.4f}")

In [10]:
from sklearn.pipeline import Pipeline

In [13]:
mlflow.set_experiment('XGBoost')

2026/05/03 11:47:19 INFO mlflow.tracking.fluent: Experiment with name 'XGBoost' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/01310a65053749a0b3439d39296a6cdf', creation_time=1777808839958, experiment_id='2', last_update_time=1777808839958, lifecycle_stage='active', name='XGBoost', tags={}, trace_location=None, workspace='default'>

In [14]:
from xgboost import XGBClassifier
from sklearn.impute import SimpleImputer

xgb_pipeline_base = Pipeline([
    ('drop_nan', DropMissingFeatures(threshold=0.9)),
    ('cat_enc', CategoricalEncoder()),
    ('imputer', SimpleImputer(strategy='median')),
    ('model', XGBClassifier(
        n_estimators=500,       
        learning_rate=0.05,     
        max_depth=6,            
        tree_method='hist',  
        random_state=42,
        scale_pos_weight=25    
    ))
])


run_experiment(xgb_pipeline_base, "XGB_Baseline")

2026/05/03 11:49:52 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/03 11:49:52 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Run 'XGB_Baseline' completed. AUC: 0.9462
🏃 View run XGB_Baseline at: https://dagshub.com/mkekn23/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/2/runs/c5cd61afa9984e5680b6dcaa9f211815
🧪 View experiment at: https://dagshub.com/mkekn23/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/2


In [23]:
# საწვრთნელი მონაცემების AUC
y_train_probs = xgb_pipeline_base.predict_proba(X_train)[:, 1]
train_auc = roc_auc_score(y_train, y_train_probs)

# სატესტო მონაცემების AUC
y_test_probs = xgb_pipeline_base.predict_proba(X_test)[:, 1]
test_auc = roc_auc_score(y_test, y_test_probs)

print(f"Train AUC: {train_auc:.4f}")
print(f"Test AUC: {test_auc:.4f}")
print(f"Difference: {train_auc - test_auc:.4f}")

Train AUC: 0.9686
Test AUC: 0.9487
Difference: 0.0199


In [20]:
from xgboost import XGBClassifier
from sklearn.impute import SimpleImputer

xgb_pipeline_base = Pipeline([
    ('time_enc', TimeFeaturesEncoder()),     
    ('aggr', GroupAggregationTransformer()),
    ('drop_nan', DropMissingFeatures(threshold=0.9)),
    ('cat_enc', CategoricalEncoder()),
    ('imputer', SimpleImputer(strategy='median')),
    ('model', XGBClassifier(
        n_estimators=500,       
        learning_rate=0.05,     
        max_depth=6,            
        tree_method='hist',  
        random_state=42,
        scale_pos_weight=25    
    ))
])


run_experiment(xgb_pipeline_base, "XGB_Enhanced")

2026/05/03 12:05:41 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/03 12:05:42 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Run 'XGB_Enhanced' completed. AUC: 0.9487
🏃 View run XGB_Enhanced at: https://dagshub.com/mkekn23/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/2/runs/462ce63a0688424faa56bc534b5fd400
🧪 View experiment at: https://dagshub.com/mkekn23/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/2


In [21]:
from xgboost import XGBClassifier

xgb_enhanced_params = {
    'n_estimators': 1000,
    'learning_rate': 0.02,
    'max_depth': 8,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'tree_method': 'hist', 
    'scale_pos_weight': 25,   
    'random_state': 42
}

xgb_pipeline_tuned = Pipeline([
    ('time_enc', TimeFeaturesEncoder()),           
    ('aggr', GroupAggregationTransformer()),     
    ('drop_nan', DropMissingFeatures(threshold=0.9)),
    ('cat_enc', CategoricalEncoder()),
    ('imputer', SimpleImputer(strategy='median')),
    ('model', XGBClassifier(**xgb_enhanced_params))
])


run_experiment(xgb_pipeline_tuned, "XGB_Enhanced_Tuned", params=xgb_enhanced_params)

2026/05/03 12:14:55 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/03 12:14:56 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Run 'XGB_Enhanced_Tuned' completed. AUC: 0.9641
🏃 View run XGB_Enhanced_Tuned at: https://dagshub.com/mkekn23/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/2/runs/9997bff0b62746c48d366d2bee5b485c
🧪 View experiment at: https://dagshub.com/mkekn23/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/2


In [22]:
# საწვრთნელი მონაცემების AUC
y_train_probs = xgb_pipeline_tuned.predict_proba(X_train)[:, 1]
train_auc = roc_auc_score(y_train, y_train_probs)

# სატესტო მონაცემების AUC
y_test_probs = xgb_pipeline_tuned.predict_proba(X_test)[:, 1]
test_auc = roc_auc_score(y_test, y_test_probs)

print(f"Train AUC: {train_auc:.4f}")
print(f"Test AUC: {test_auc:.4f}")
print(f"Difference: {train_auc - test_auc:.4f}")

Train AUC: 0.9908
Test AUC: 0.9641
Difference: 0.0266


In [24]:
from xgboost import XGBClassifier

xgb_enhanced_params = {
    'n_estimators': 1000,
    'learning_rate': 0.02,
    'max_depth': 8,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'tree_method': 'hist', 
    'scale_pos_weight': 25,   
    'random_state': 42
}

xgb_pipeline_tuned = Pipeline([
    ('time_enc', TimeFeaturesEncoder()),           
    ('aggr', GroupAggregationTransformer()),     
    ('drop_nan', DropMissingFeatures(threshold=0.85)),
    ('cat_enc', CategoricalEncoder()),
    ('imputer', SimpleImputer(strategy='median')),
    ('model', XGBClassifier(**xgb_enhanced_params))
])


run_experiment(xgb_pipeline_tuned, "XGB_Enhanced_Tuned_lessDropingThreshold", params=xgb_enhanced_params)

2026/05/03 14:37:32 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/03 14:37:33 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Run 'XGB_Enhanced_Tuned_lessDropingThreshold' completed. AUC: 0.9645
🏃 View run XGB_Enhanced_Tuned_lessDropingThreshold at: https://dagshub.com/mkekn23/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/2/runs/9f6e28595f6d457a8dcd5f860e27c0a4
🧪 View experiment at: https://dagshub.com/mkekn23/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/2


In [25]:
# საწვრთნელი მონაცემების AUC
y_train_probs = xgb_pipeline_tuned.predict_proba(X_train)[:, 1]
train_auc = roc_auc_score(y_train, y_train_probs)

# სატესტო მონაცემების AUC
y_test_probs = xgb_pipeline_tuned.predict_proba(X_test)[:, 1]
test_auc = roc_auc_score(y_test, y_test_probs)

print(f"Train AUC: {train_auc:.4f}")
print(f"Test AUC: {test_auc:.4f}")
print(f"Difference: {train_auc - test_auc:.4f}")

Train AUC: 0.9918
Test AUC: 0.9645
Difference: 0.0273
